In [ ]:
import os
import cv2
from matplotlib import pyplot as plt
import numpy as np
import open3d as o3d
import math


def data_path_loader(debugMode=False):
    if debugMode == True:
        # Paths
        calib_path = "~/34759_final_project_rect/calib_cam_to_cam.txt"
        img_02_path = "~/34759_final_project_rect/seq_03/image_02/data/"
        img_03_path = "~/34759_final_project_rect/seq_03/image_03/data/"
        bboxes_2D_path = "~/labels_px/"
        video_path = "~/Video/"
        bboxes_3D_xyz_path = "~/bbox_3D_xyz/"
        bboxes_3D_uv_path = "~/bbox_3D_uv/"
    else:
        None  

    return calib_path, img_02_path, img_03_path, bboxes_2D_path, bboxes_3D_xyz_path, bboxes_3D_uv_path


def get_calibration_data(calib_path, show=False):
    file = open(calib_path, "r")
    lines = file.readlines()

    # Projection matrices (rectified cam00 coordinates -> image plane of cam0X)
    line = lines[25].strip().split(" ")[1:]
    P_rect_02 = np.array([float(string) for string in line]).reshape((3, 4))

    line = lines[33].strip().split(" ")[1:]
    P_rect_03 = np.array([float(string) for string in line]).reshape((3, 4))

    # New intrinsic (K_rect_0X) and extrinsic (T_rect_0X) matrices, decomposed from P_rect_0X
    K_rect_02, _, T_rect_02, _, _, _, _ = cv2.decomposeProjectionMatrix(P_rect_02)
    T_rect_02 /= T_rect_02[3] # to get homogenous expression

    K_rect_03, _, T_rect_03, _, _, _, _ = cv2.decomposeProjectionMatrix(P_rect_03)
    T_rect_03 /= T_rect_03[3] # to get homogenous expression

    # Rotation matrices (unrectified cam0X coordinates -> rectified cam0X coordinates)
    line = lines[24].strip().split(" ")[1:]
    R_rect_02 = np.array([float(string) for string in line]).reshape((3, 3))
    R_rect_02 = np.pad(R_rect_02, ((0, 1), (0, 1)), mode='constant') # to get 4x4 shape
    R_rect_02[3, 3] = 1.0

    line = lines[32].strip().split(" ")[1:]
    R_rect_03 = np.array([float(string) for string in line]).reshape((3, 3))
    R_rect_03 = np.pad(R_rect_03, ((0, 1), (0, 1)), mode='constant') # to get 4x4 shape
    R_rect_03[3, 3] = 1.0

    file.close()

    if show is True:
        print(f"P_rect_02: \n {P_rect_02} \n")
        print(f"P_rect_03: \n {P_rect_03} \n")
        print(f"K_rect_02: \n {K_rect_02} \n")
        print(f"K_rect_03: \n {K_rect_03} \n")
        print(f"T_rect_02: \n {T_rect_02} \n")
        print(f"T_rect_03: \n {T_rect_03} \n")
        print(f"R_rect_02: \n {R_rect_02} \n")
        print(f"R_rect_03: \n {R_rect_03} \n")

    return P_rect_02, P_rect_03, K_rect_02, K_rect_03, T_rect_02, T_rect_03, R_rect_02, R_rect_03


def get_2D_bboxes(bboxes_2D_path, bbox_2D_name, show=False):
    bboxes = []
    with open(os.path.join(bboxes_2D_path, bbox_2D_name), "r") as file:
        for line in file:
            bbox = list(map(float, line.strip().split()[2:]))
            bboxes.append(bbox)
    bboxes_center_length = np.array(bboxes)
    x_min = bboxes_center_length[:, 0] - bboxes_center_length[:, 2] / 2
    y_min = bboxes_center_length[:, 1] - bboxes_center_length[:, 3] / 2
    x_max = bboxes_center_length[:, 0] + bboxes_center_length[:, 2] / 2
    y_max = bboxes_center_length[:, 1] + bboxes_center_length[:, 3] / 2
    bboxes_2D = np.stack((x_min, y_min, x_max, y_max), axis=1)

    if show is True:
        print(bboxes_2D)

    return bboxes_2D


def compute_disparity_map(img_02_gray, img_03_gray, show=False):
    # Disparity parameters
    min_disp = 6
    num_disp = 6 * 16
    block_size = 7 #This parameter is worth tweaking, [1, 5, 7, 9, 11]
    #stereo = cv2.StereoSGBM_create(numDisparities = num_disp, blockSize = block_size) #take semi-global block matching (SGBM) instead of block matching (BM) to get a more complete disparity map
    stereo = cv2.StereoSGBM_create(0, numDisparities=num_disp, blockSize=block_size, P1=600, P2=2400, mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY)
    stereo.setMinDisparity(min_disp)
    stereo.setDisp12MaxDiff(500)
    stereo.setUniquenessRatio(1)
    stereo.setSpeckleRange(5)
    stereo.setSpeckleWindowSize(101)

    disparity_map = stereo.compute(img_02_gray, img_03_gray).astype(np.float32) / 16.0

    disparity_map = cv2.medianBlur(disparity_map, 5)

    if show is True:
        plt.figure(figsize=(18,18))
        #plt.title("Disparity Map")
        plt.imshow(disparity_map, cmap='gray')

    return disparity_map


def compute_depth_map(disparity_map, K_rect_02, T_rect_02, T_rect_03, show=False):
    # Focal length of cam02 and cam03
    focal_length = K_rect_02[0, 0]
    
    # Distance between cam02 and cam03
    baseline = np.abs(T_rect_02[0] - T_rect_03[0])[0]
    
    #Replace all instances of 0 and -1 disparity with a small minimum value (to avoid div by 0 or negatives)
    #disparity_map[disparity_map <= 0] = 1e-5
    
    # Calculate depth from disparity
    depth_map = focal_length * baseline / disparity_map

    #depth_map = cv2.medianBlur(depth_map, 5)

    if show is True:
        plt.figure(figsize=(18,18))
        #plt.title("Depth Map")
        plt.imshow(depth_map, cmap='gray')

    return depth_map


def compute_3D_bboxes(bboxes_2D, depth_map, K_rect_02, T_rect_02, T_rect_03, disparity_map, percentile=0.48, show=False):
    # Create Q-matrix
    baseline = T_rect_02[0] - T_rect_03[0]
    Q_02 = np.array([
            [1, 0, 0, -K_rect_02[0, 2]],
            [0, 1, 0, -K_rect_02[1, 2]],
            [0, 0, 0, K_rect_02[0, 0]],
            [0, 0, -1/baseline.item(), 0]])
    
    # Map all points from 2D image plane to 3D space
    XYZ = cv2.reprojectImageTo3D(disparity_map, Q_02)

    bboxes_3D = np.empty((1, 8, 3))
    median_depths = []
    # Iterate through all given 2D bounding boxes
    for bbox_2D in bboxes_2D:
        # Get inner pixels of bounding box
        bbox_u_min = math.ceil(bbox_2D[0])
        bbox_u_max = math.floor(bbox_2D[2])
        bbox_v_min = math.ceil(bbox_2D[1])
        bbox_v_max = math.floor(bbox_2D[3])

        # Slice 3D points in 2D bounding box area
        XYZ_region_bbox = XYZ[bbox_v_min : bbox_v_max, bbox_u_min : bbox_u_max]
        
        # Compute median depth (z-coordinate)
        median_depth = np.median(XYZ_region_bbox[:, :, 2])
        median_depths.append(median_depth)

        # Outlier elimination through discarding points that are not close to median value
        points = XYZ_region_bbox.reshape(-1, 3)
        points_sorted = points[points[:, 2].argsort()] # argsort creates index array that is sorted w.r.t z-coordinate
        num_points = points.shape[0]
        inlier_points = points_sorted[math.ceil(percentile * num_points) : math.floor((1 - percentile) * num_points)]

        # Get 3D bounding box dimensions
        bbox_x_min = np.min(inlier_points[:, 0])
        bbox_x_max = np.max(inlier_points[:, 0])
        bbox_y_min = np.min(inlier_points[:, 1])
        bbox_y_max = np.max(inlier_points[:, 1])
        bbox_z_min = np.min(inlier_points[:, 2])
        bbox_z_max = np.max(inlier_points[:, 2])
        bbox_3D = np.array([
            [bbox_x_min, bbox_y_min, bbox_z_min],
            [bbox_x_max, bbox_y_min, bbox_z_min],
            [bbox_x_max, bbox_y_max, bbox_z_min],
            [bbox_x_min, bbox_y_max, bbox_z_min],
            [bbox_x_min, bbox_y_min, bbox_z_max],
            [bbox_x_max, bbox_y_min, bbox_z_max],
            [bbox_x_max, bbox_y_max, bbox_z_max],
            [bbox_x_min, bbox_y_max, bbox_z_max]
        ])

        # Append to list of 3D bboxes
        bboxes_3D = np.concatenate([bboxes_3D, bbox_3D[np.newaxis, :]], axis=0)

    if show is True:
        print(f"3D bboxes in cam02 frame, dimensions: (bbox x bbox points x cam02 coordinates X/Y/Z): \n {bboxes_3D[1:]} \n")
        print()
        print(f"Median Depth: {np.array(median_depths)} \n")
        print()
            
    return bboxes_3D[1:], np.array(median_depths)


def show_3D_bboxes_image_plane(bboxes_3D, P_rect_02, R_rect_02, img_02, median_depths, show=False):
    # Iterate through all 3D bounding boxes
    bboxes_3D_uv = np.empty((1, 8, 2))
    for bbox_3D, median_depth in zip(bboxes_3D, median_depths):
        # Extend to homogenous coordinates
        XYZW = np.hstack((bbox_3D, np.ones((bbox_3D.shape[0], 1))))
        # Project to image plane
        UVW = P_rect_02 @ R_rect_02 @ XYZW.T
        UVW /= UVW[2, :] # to get homogenous expression
        UV = np.round(UVW[:2]).astype(int)

        # Draw bounding box on image plane
        cv2.line(img_02, (UV[0, 0], UV[1, 0]), (UV[0, 1], UV[1, 1]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 1], UV[1, 1]), (UV[0, 2], UV[1, 2]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 2], UV[1, 2]), (UV[0, 3], UV[1, 3]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 3], UV[1, 3]), (UV[0, 0], UV[1, 0]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 4], UV[1, 4]), (UV[0, 5], UV[1, 5]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 5], UV[1, 5]), (UV[0, 6], UV[1, 6]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 6], UV[1, 6]), (UV[0, 7], UV[1, 7]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 7], UV[1, 7]), (UV[0, 4], UV[1, 4]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 0], UV[1, 0]), (UV[0, 4], UV[1, 4]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 1], UV[1, 1]), (UV[0, 5], UV[1, 5]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 2], UV[1, 2]), (UV[0, 6], UV[1, 6]), color=(255, 0, 0), thickness=2)
        cv2.line(img_02, (UV[0, 3], UV[1, 3]), (UV[0, 7], UV[1, 7]), color=(255, 0, 0), thickness=2)

        # Draw median depth for each bounding box
        cv2.putText(img_02, '{0:.2f} m'.format(median_depth), (UV[0, 4]-2, UV[1, 4]-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 0, 0), 2, cv2.LINE_AA)

        # Append to list of 3D bboxes in image plane
        bboxes_3D_uv = np.concatenate([bboxes_3D_uv, np.transpose(UV)[np.newaxis, :]], axis=0)

    if show is True:
        #print(f"3D bboxes on image plane, dimensions: (bbox x bbox points x image coordinates U/V): \n {bboxes_3D_uv[1:]} \n")
        #print()
        plt.figure(figsize=(18,18))
        #plt.title("Left Image with 3D Bounding Boxes")
        plt.imshow(img_02)

    return bboxes_3D_uv[1:], img_02


def generate_3D_point_cloud(img_02, disparity_map, K_rect_02, bboxes_3D, show=True):
    # Convert OpenCV images to Open3D images
    o3d_color = o3d.geometry.Image(img_02)
    o3d_depth = o3d.geometry.Image(disparity_map)
    # Create point cloud by utilizing RGBD image and camera intrinsics
    rgbd_image = o3d.geometry.RGBDImage.create_from_color_and_depth(o3d_color, o3d_depth, convert_rgb_to_intensity = False)
    camera_intrinsics = o3d.camera.PinholeCameraIntrinsic(img_02.shape[1], img_02.shape[0], K_rect_02[0, 0], K_rect_02[1, 1], K_rect_02[0, 2], K_rect_02[1, 2])
    point_cloud = o3d.geometry.PointCloud.create_from_rgbd_image(rgbd_image, camera_intrinsics)
    point_cloud.transform([[1, 0, 0, 0], [0, -1, 0, 0], [0, 0, 1, 0], [0, 0, 0, 1]]) # Flip it, otherwise upside-down

    # Add 3D bboxes
    all_points = []
    all_lines = []
    all_colors = []

    for bbox_3D in bboxes_3D:
        points = bbox_3D
        lines = [[0, 1], [1, 2], [2, 3], [0, 3],
                [4, 5], [5, 6], [6, 7], [4, 7],
                [0, 4], [1, 5], [2, 6], [3, 7]]
        colors = [[1, 0, 0] for _ in range(len(lines))]
        all_points.extend(points)
        all_lines.extend(lines)
        all_colors.extend(colors)

    line_set = o3d.geometry.LineSet()
    line_set.points = o3d.utility.Vector3dVector(all_points)
    line_set.lines = o3d.utility.Vector2iVector(all_lines)
    line_set.colors = o3d.utility.Vector3dVector(all_colors)

    if show is True:
        o3d.visualization.draw_geometries([point_cloud])
        #o3d.visualization.draw_geometries([point_cloud, line_set])


def output_write_file(bboxes_3D_path, txt_name, bboxes_3D):
    np.savetxt(os.path.join(bboxes_3D_path, txt_name), bboxes_3D, delimiter=' ')


def record_video(video_frames, show=False):
    if show is True:
        frame_height, frame_width, _ = video_frames[0].shape
        video_path = "Video/3D_Tracking2.mp4"
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        fps = 20
        video_writer = cv2.VideoWriter(video_path, fourcc, fps, (frame_width, frame_height))
        for frame in video_frames:
            frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)
            video_writer.write(frame)
        video_writer.release()
    



###### Main #####
video_frames = []

# Data path loader
calib_path, img_02_path, img_03_path, bboxes_2D_path, bboxes_3D_xyz_path, bboxes_3D_uv_path = data_path_loader(debugMode=True)

### a) Extract calibration data from given KITTI file 'calib_cam_to_cam.txt' and transfrom data to our needs
P_rect_02, P_rect_03, K_rect_02, K_rect_03, T_rect_02, T_rect_03, R_rect_02, R_rect_03 = get_calibration_data(calib_path, show=False)

# Get a sorted list of all images
files_02 = sorted(os.listdir(img_02_path))
files_03 = sorted(os.listdir(img_03_path))
files_bbox = sorted(os.listdir(bboxes_2D_path))
# Loop through whole dataset
for img_02_name, img_03_name, bbox_2D_name in zip(files_02, files_03, files_bbox):
    img_02 = cv2.imread(os.path.join(img_02_path, img_02_name))
    img_03 = cv2.imread(os.path.join(img_03_path, img_03_name))
    # Convert to RGB
    blue02, green02, red02 = cv2.split(img_02)
    img_02 = cv2.merge([red02, green02, blue02])
    blue03, green03, red03 = cv2.split(img_03)
    img_03 = cv2.merge([red03, green03, blue03])
    # Convert to grayscale
    img_02_gray = cv2.cvtColor(img_02, cv2.COLOR_RGB2GRAY)
    img_03_gray = cv2.cvtColor(img_03, cv2.COLOR_RGB2GRAY)

    ### b) Get 2D bboxes provided by previous tracking algorithm
    bboxes_2D = get_2D_bboxes(bboxes_2D_path, bbox_2D_name, show=False)

    ### c) Compute disparity map using stereo image pairs
    disparity_map = compute_disparity_map(img_02_gray, img_03_gray, show=False)
    #cv2.imwrite(os.path.join(disparity_path, img_02_name), disparity_map.astype(np.uint8))

    ### d) Compute depth map by utilizing camera intrinsics and stereo camera baseline information
    depth_map = compute_depth_map(disparity_map, K_rect_02, T_rect_02, T_rect_03, show=False)

    ### e) Calculate 3D points of the given 2D bounding box areas
    bboxes_3D, median_depths = compute_3D_bboxes(bboxes_2D, depth_map, K_rect_02, T_rect_02, T_rect_03, disparity_map, percentile=0.30, show=False)

    ### f) Show 3D bounding box on cam02 image plane
    bboxes_3D_uv, img_02 = show_3D_bboxes_image_plane(bboxes_3D, P_rect_02, R_rect_02, img_02, median_depths, show=False)
    
    ### g) Create 3D point cloud
    generate_3D_point_cloud(img_02, disparity_map, K_rect_02, bboxes_3D, show=False)
    
    ### h) Save 3D bounding boxes in xyz and uv coordinates in .txt files
    txt_name = img_02_name.replace(".png", ".txt")
    output_write_file(bboxes_3D_xyz_path, txt_name, bboxes_3D.reshape(-1, 3))
    output_write_file(bboxes_3D_uv_path, txt_name, bboxes_3D_uv.reshape(-1, 2))
    
    ### i) Record video sequence
    video_frames.append(img_02)
record_video(video_frames, show=True)
